# Cuarta Entrega (IL1.4) - Evaluación, Monitoreo y Observabilidad

Hasta ahora hemos desarrollado nuestro chatbot en `ChatBot.ipynb`, ajustado sus prompts en `Segunda_Entrega.ipynb`, y le hemos dado conocimiento de los manuales de la empresa mediante RAG en `Tercera_Entrega.ipynb`.

En la etapa final de este ciclo (RA1), debemos implementar estrategias para **observar, medir y evaluar** el desempeño del agente. Para ello utilizaremos **LangSmith**, la plataforma de trazabilidad de LangChain.

In [ ]:
import os
from dotenv import load_dotenv

# Cargar variables de entorno
load_dotenv("/Users/RamonOsorio/Desktop/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/.env")

# Verificar configuración de LangSmith
langsmith_activo = os.environ.get("LANGCHAIN_TRACING_V2") or os.environ.get("LANGSMITH_TRACING")
proyecto = os.environ.get("LANGCHAIN_PROJECT") or os.environ.get("LANGSMITH_PROJECT")

print(f"Trazabilidad con LangSmith: {'Activada' if langsmith_activo == 'true' else 'Desactivada'}")
print(f"Proyecto LangSmith: {proyecto}")

if not langsmith_activo == 'true':
    print("\n⚠️ ADVERTENCIA: Para que la trazabilidad funcione, asegúrate de tener las siguientes variables en tu archivo .env:")
    print("LANGCHAIN_TRACING_V2=true")
    print("LANGCHAIN_API_KEY=tu_api_key_de_langsmith")
    print("LANGCHAIN_PROJECT=InvermarBot")

### 1. Invocación de la Cadena RAG con Trazabilidad
Al estar LangSmith configurado en el archivo `.env`, cada ejecución del LLM se envía automáticamente al dashboard de LangSmith. Esto nos permite ver los tokens consumidos, la latencia y la estructura exacta del prompt que llegó al modelo.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

token = os.environ.get("GITHUB_TOKEN")
base_url = os.environ.get("OPENAI_BASE_URL", "https://models.inference.ai.azure.com")

# (Reconstruimos rápidamente el RAG de la Tercera Entrega)
embeddings = OpenAIEmbeddings(base_url=base_url, api_key=token, model="text-embedding-3-small")
politicas = ["Aguinaldo Fiestas Patrias: $85.000 líquidos.", "Finiquito: 10 días hábiles en Notaría."]
vectorstore = FAISS.from_texts(politicas, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

llm = ChatOpenAI(base_url=base_url, api_key=token, model="gpt-4o-mini", temperature=0)

template = "Contexto:\n{contexto}\n\nPregunta: {pregunta}\n\nRespuesta:"
prompt_rag = ChatPromptTemplate.from_template(template)

def format_docs(docs): return "\n".join(d.page_content for d in docs)

cadena_rag = (
    {"contexto": retriever | format_docs, "pregunta": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)

# Hacer una consulta de prueba
pregunta = "¿Cuándo me pagan el aguinaldo?"
print(f"Enviando consulta: '{pregunta}'")

# Esta ejecución se trazará automáticamente en LangSmith
respuesta = cadena_rag.invoke(pregunta)
print(f"\nRespuesta: {respuesta}")
print("\n✓ ¡Revisa tu dashboard en smith.langchain.com para ver la traza de esta ejecución!")

### 2. Evaluación Cualitativa de Respuestas (Evaluador LLM-as-a-Judge)
Para evaluar automáticamente si las respuestas del bot son correctas y útiles, podemos usar otro LLM como "Juez". Este modelo leerá la respuesta y la calificará.

In [ ]:
# Prompt para el modelo evaluador
prompt_evaluador = ChatPromptTemplate.from_template("""
Eres un evaluador de calidad para un chatbot de Recursos Humanos.
Debes evaluar si la respuesta del bot es correcta basándote en el contexto entregado.

Contexto real: {contexto}
Pregunta del usuario: {pregunta}
Respuesta del bot: {respuesta_bot}

Evalúa la respuesta del bot respondiendo ÚNICAMENTE con una de las siguientes opciones:
[CORRECTO]: La respuesta se basa fielmente en el contexto y responde la pregunta.
[INCORRECTO]: La respuesta contradice el contexto o inventa información.
[INCOMPLETO]: La respuesta omite información crucial del contexto.
""")

# Cadena evaluadora
evaluador_llm = ChatOpenAI(base_url=base_url, api_key=token, model="gpt-4o-mini", temperature=0)
cadena_evaluadora = prompt_evaluador | evaluador_llm | StrOutputParser()

# Prueba de evaluación
evaluacion = cadena_evaluadora.invoke({
    "contexto": "El aguinaldo es de $85.000",
    "pregunta": "¿Cuánto es el aguinaldo?",
    "respuesta_bot": respuesta
})

print("Evaluación automática del Bot Juez:")
print(f"Calificación: {evaluacion}")